In [ ]:
# VQA Project — Data Loading & Smoke Test
# Verify local data files load correctly and run a 

quick model forward pass.

In [ ]:
import json
import os
import sys
import matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image
from collections import Counter

# Navigate to project root so all relative paths in config work correctly
if Path.cwd().name == "notebooks":
    os.chdir("..")
sys.path.insert(0, ".")

import yaml
with open("configs/config.yaml") as f:
    cfg = yaml.safe_load(f)
print("Config loaded. Project root:", Path.cwd())

In [ ]:
with open(cfg["data"]["questions_train"]) as f:
    questions_data = json.load(f)

with open(cfg["data"]["annotations_train"]) as f:
    annotations_data = json.load(f)

questions   = questions_data["questions"]
annotations = annotations_data["annotations"]

ann_by_qid = {a["question_id"]: a for a in annotations}
samples = []
for q in questions:
    ann = ann_by_qid.get(q["question_id"])
    if ann:
        samples.append({
            "image_id":    q["image_id"],
            "question_id": q["question_id"],
            "question":    q["question"],
            "answers":     ann["answers"],
            "answer_type": ann["answer_type"],
        })

print(f"Total samples: {len(samples)}")
print(f"Example sample:\n{samples[0]}")

In [ ]:
# Answer type distribution
type_counts = Counter(s["answer_type"] for s in samples)
print("Answer type distribution:")
for k, v in type_counts.items():
    print(f"  {k}: {v} ({100*v/len(samples):.1f}%)")

# Top 20 most common answers
all_answers = [a["answer"] for s in samples for a in s["answers"]]
top_answers = Counter(all_answers).most_common(20)
print("\nTop 20 answers:")
for ans, count in top_answers:
    print(f"  {ans}: {count}")

In [ ]:
def load_image(image_id, split="train"):
    img_dir = Path(cfg["data"][f"images_{split}"])
    filename = f"COCO_{split}2014_{image_id:012d}.jpg"
    return Image.open(img_dir / filename).convert("RGB")

img = load_image(samples[0]["image_id"], split="train")
print(f"Image size: {img.size}")

Path(cfg["paths"]["eda_plots_dir"]).mkdir(parents=True, exist_ok=True)
plt.imshow(img)
plt.title(samples[0]["question"])
plt.axis("off")
plt.savefig(Path(cfg["paths"]["eda_plots_dir"]) / "sample_image.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(20, 10))
for i, ax in enumerate(axes.flatten()):
    s = samples[i]
    img = load_image(s["image_id"], split="train")
    top_answer = Counter(a["answer"] for a in s["answers"]).most_common(1)[0][0]
    ax.imshow(img)
    ax.set_title(f"Q: {s['question']}\nA: {top_answer}", fontsize=8)
    ax.axis("off")
plt.tight_layout()
plt.savefig(Path(cfg["paths"]["eda_plots_dir"]) / "sample_grid.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
a_lengths = [len(a["answer"].split()) for s in samples for a in s["answers"]]
plt.figure(figsize=(10, 4))
plt.hist(a_lengths, bins=20, color="steelblue", edgecolor="white")
plt.xlabel("Answer Length (words)")
plt.ylabel("Count")
plt.title("Answer Length Distribution")
plt.tight_layout()
plt.savefig(Path(cfg["paths"]["eda_plots_dir"]) / "answer_length_dist.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
import torch
from src.models.vqa_model import VQAModel

# cfg already loaded above — no need to reload configs/config.yaml
model = VQAModel(cfg, vocab_size=3129, mode="multimodal")

dummy_batch = {
    "image_tensor":   torch.randn(2, 3, 224, 224),
    "question_ids":   torch.randint(0, 1000, (2, 30)),
    "attention_mask": torch.ones(2, 30, dtype=torch.long),
}

with torch.no_grad():
    output = model(dummy_batch)

print("Output keys:", list(output.keys()))
print("Answer logits shape:", output["answer_logits"].shape)
print("Top-3 answers:", output["top3_answers"])
print("Confidence:", output["confidence"])
print("Smoke test passed ✓")